1. Загрузите данные на свой жёсткий диск.

2. Считайте их в polars.

3. Посчитайте уникальное количество кодов категорий двумя разными способами.

4. Замените отсутствующие коды категорий значением по выбору и убедитесь, что описательная статистика поменялась.

5. На основе оконных функций постройте поле, состоящее из максимальной цены в разбивке по коду категории.

6. Постройте новое поле, принимающее значение True, если цена больше средней цены, и False в ином случае.

In [ ]:
import polars as pl
# 1. Загрузка данных
events = pl.read_csv('events.csv')
events

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
str,str,i64,i64,str,str,f64,i64,str
"""2020-09-24 11:57:06 UTC""","""view""",1996170,2144415922528452715,"""electronics.telephone""",null,31.9,1515915625519388267,"""LJuJVLEjPT"""
"""2020-09-24 11:57:26 UTC""","""view""",139905,2144415926932472027,"""computers.components.cooler""","""zalman""",17.16,1515915625519380411,"""tdicluNnRY"""
"""2020-09-24 11:57:27 UTC""","""view""",215454,2144415927158964449,null,null,9.81,1515915625513238515,"""4TMArHtXQy"""
"""2020-09-24 11:57:33 UTC""","""view""",635807,2144415923107266682,"""computers.peripherals.printer""","""pantum""",113.81,1515915625519014356,"""aGFYrNgC08"""
"""2020-09-24 11:57:36 UTC""","""view""",3658723,2144415921169498184,null,"""cameronsino""",15.87,1515915625510743344,"""aa4mmk0kwQ"""
…,…,…,…,…,…,…,…,…
"""2021-02-28 23:55:01 UTC""","""view""",953226,2144415927553229037,null,null,219.94,1515915625611023730,"""FRLqIttxKU"""
"""2021-02-28 23:58:05 UTC""","""view""",1715907,2144415927049912542,"""electronics.video.tv""","""starwind""",80.03,1515915625611024014,"""g6WqPf50Ma"""
"""2021-02-28 23:58:09 UTC""","""view""",4170534,2144415939364389423,"""electronics.clocks""","""amazfit""",64.92,1515915625611024020,"""xNIJBqZdkd"""


In [ ]:
# 2. Статистика по данным
events.describe()

statistic,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
str,str,str,f64,f64,str,str,f64,f64,str
"""count""","""885129""","""885129""",885129.0,885129.0,"""648910""","""672765""",885129.0,885129.0,"""884964"""
"""null_count""","""0""","""0""",0.0,0.0,"""236219""","""212364""",0.0,0.0,"""165"""
"""mean""",null,null,1.9066e6,2.1444e18,null,null,146.328713,1.5159e18,null
"""std""",null,null,1.4587e6,6.1651e14,null,null,296.807683,3.5549e7,null
"""min""","""2020-09-24 11:57:06 UTC""","""cart""",102.0,2.1444e18,"""accessories.bag""","""a-data""",0.22,1.5159e18,"""000AMhYaQu"""
"""25%""",null,null,698803.0,2.1444e18,null,null,26.46,1.5159e18,null
"""50%""",null,null,1.452883e6,2.1444e18,null,null,65.71,1.5159e18,null
"""75%""",null,null,3.721194e6,2.1444e18,null,null,190.49,1.5159e18,null
"""max""","""2021-02-28 23:59:09 UTC""","""view""",4.18388e6,2.2278e18,"""stationery.stapler""","""zyxel""",64771.06,1.5159e18,"""zzzYMiLcf7"""


In [ ]:
# 3. Подсчет уникальных кодов категорий разными способами
print(f"Через Unique: {events['category_code'].n_unique()}")
print("-"*50)
print(f"Через select: {events.select(pl.approx_n_unique('category_code'))}")
print("-"*50)
print(f"Через GroupBy: {events.group_by('category_code').len().shape[0]}")

Через Unique: 108
--------------------------------------------------
Через select: shape: (1, 1)
┌───────────────┐
│ category_code │
│ ---           │
│ u32           │
╞═══════════════╡
│ 108           │
└───────────────┘
--------------------------------------------------
Через GroupBy: 108


In [ ]:
# 4. Замените отсутствующие коды категорий значением по выбору и убедитесь, что описательная статистика поменялась.
events_filled = events.fill_null('unknown')
events_filled.describe()

statistic,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
str,str,str,f64,f64,str,str,f64,f64,str
"""count""","""885129""","""885129""",885129.0,885129.0,"""885129""","""885129""",885129.0,885129.0,"""885129"""
"""null_count""","""0""","""0""",0.0,0.0,"""0""","""0""",0.0,0.0,"""0"""
"""mean""",null,null,1.9066e6,2.1444e18,null,null,146.328713,1.5159e18,null
"""std""",null,null,1.4587e6,6.1651e14,null,null,296.807683,3.5549e7,null
"""min""","""2020-09-24 11:57:06 UTC""","""cart""",102.0,2.1444e18,"""accessories.bag""","""a-data""",0.22,1.5159e18,"""000AMhYaQu"""
"""25%""",null,null,698803.0,2.1444e18,null,null,26.46,1.5159e18,null
"""50%""",null,null,1.452883e6,2.1444e18,null,null,65.71,1.5159e18,null
"""75%""",null,null,3.721194e6,2.1444e18,null,null,190.49,1.5159e18,null
"""max""","""2021-02-28 23:59:09 UTC""","""view""",4.18388e6,2.2278e18,"""unknown""","""zyxel""",64771.06,1.5159e18,"""zzzYMiLcf7"""


In [ ]:
# Отфильтруйте теперь только те значения в столбике категроии коде которые не были заполнены и убедитесь, что там нет пропусков.
events_filled.filter(pl.col('category_code') == 'unknown')


event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
str,str,i64,i64,str,str,f64,i64,str
"""2020-09-24 11:57:27 UTC""","""view""",215454,2144415927158964449,"""unknown""","""unknown""",9.81,1515915625513238515,"""4TMArHtXQy"""
"""2020-09-24 11:57:36 UTC""","""view""",3658723,2144415921169498184,"""unknown""","""cameronsino""",15.87,1515915625510743344,"""aa4mmk0kwQ"""
"""2020-09-24 11:58:25 UTC""","""view""",657859,2144415939431498289,"""unknown""","""unknown""",34.17,1515915625519320570,"""HEl15U7JVy"""
"""2020-09-24 11:58:34 UTC""","""view""",811491,2144415926370435276,"""unknown""","""ritmix""",33.32,1515915625356205647,"""aFLc6y9kn4"""
"""2020-09-24 11:58:54 UTC""","""view""",811491,2144415926370435276,"""unknown""","""ritmix""",33.32,1515915625356205647,"""aFLc6y9kn4"""
…,…,…,…,…,…,…,…,…
"""2021-02-28 23:36:07 UTC""","""view""",775400,2144415921169498184,"""unknown""","""unknown""",15.08,1515915625611022053,"""TNWPE1GfY9"""
"""2021-02-28 23:38:11 UTC""","""view""",1022663,2144415943961346730,"""unknown""","""cameronsino""",35.24,1515915625611022253,"""iERbMFYhhx"""
"""2021-02-28 23:39:41 UTC""","""view""",1502502,2144415933794353554,"""unknown""","""unknown""",83.97,1515915625611022372,"""DLdrfrUb1z"""


In [ ]:
# 5. На основе оконных функций постройте поле, состоящее из максимальной цены в разбивке по коду категории.
events_filled.with_columns([
    pl.col("price")
    .max()
    .over("category_code")
    .alias("max_price_by_category")
    
]).sort(by = "max_price_by_category", descending = True)

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,max_price_by_category
str,str,i64,i64,str,str,f64,i64,str,f64
"""2020-09-24 13:25:13 UTC""","""view""",889614,2144415928970903827,"""computers.peripherals.monitor""","""hama""",45.35,1515915625519420883,"""XKUzFI5Ve8""",64771.06
"""2020-09-24 13:25:29 UTC""","""view""",889614,2144415928970903827,"""computers.peripherals.monitor""","""hama""",45.35,1515915625519420883,"""XKUzFI5Ve8""",64771.06
"""2020-09-24 13:26:59 UTC""","""view""",1148993,2144415922402623591,"""computers.peripherals.monitor""","""asus""",1057.35,1515915625399431063,"""qI2ILCw35i""",64771.06
"""2020-09-24 13:27:01 UTC""","""view""",1148993,2144415922402623591,"""computers.peripherals.monitor""","""asus""",1057.35,1515915625399431063,"""OFVAuTUNOl""",64771.06
"""2020-09-24 14:40:08 UTC""","""view""",911882,2144415922402623591,"""computers.peripherals.monitor""","""samsung""",580.56,1515915625445661251,"""13vnm8LKKi""",64771.06
…,…,…,…,…,…,…,…,…,…
"""2021-02-26 07:50:18 UTC""","""view""",565453,2144415940790452820,"""computers.gaming""","""megaopt""",1.57,1515915625609742088,"""NrI4NDhz8O""",10.14
"""2021-02-26 11:21:44 UTC""","""view""",565470,2144415940790452820,"""computers.gaming""","""megaopt""",1.57,1515915625609956098,"""u5ZITjgqjt""",10.14
"""2021-02-26 22:09:57 UTC""","""view""",565470,2144415940790452820,"""computers.gaming""","""megaopt""",1.57,1515915625610162003,"""FHb9Yy6mk4""",10.14


In [ ]:
# 6. Постройте новое поле, принимающее значение True, если цена больше средней цены, и False в ином случае.
avg_price = events_filled.select(pl.col("price").mean()).item()
print(f"Средняя цена: {avg_price}")
events_bool = events_filled.select([
    pl.col("price"),
    pl.when(pl.col("price") > avg_price)
    .then(True)
    .otherwise(False)
    .alias("is_price_above_average")
])
display(events_bool)
print(f"Количество записей с ценой выше средней: {events_bool.filter(pl.col('is_price_above_average') == True).shape[0]}")
print(f"Количество записей с ценой ниже или равной средней: {events_bool.filter(pl.col('is_price_above_average') == False).shape[0]}")

Средняя цена: 146.3287134982585


price,is_price_above_average
f64,bool
31.9,false
17.16,false
9.81,false
113.81,false
15.87,false
…,…
219.94,true
80.03,false
64.92,false


Количество записей с ценой выше средней: 268850
Количество записей с ценой ниже или равной средней: 616279
